In [ ]:
# =============================================================================
# Customer Churn Prediction — Neural Network Analysis
# =============================================================================

# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, roc_auc_score)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import os, json, csv

# reproducibility
np.random.seed(42)
tf.random.set_seed(42)

RESULTS = "results"
os.makedirs(RESULTS, exist_ok=True)

# =============================================================================
# TASK 1 — Dataset Understanding
# =============================================================================
print("\n" + "="*60)
print("TASK 1 — Dataset Understanding")
print("="*60)

df = pd.read_csv("customer_churn_nn.csv")

print(f"\n▸ Shape  : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n▸ Column dtypes:\n{df.dtypes.to_string()}")
print(f"\n▸ Missing values:\n{df.isnull().sum().to_string()}")
print(f"\n▸ Statistical summary:\n{df.describe().round(2).to_string()}")

churn_counts = df["churn"].value_counts()
print(f"\n▸ Target distribution:\n{churn_counts.to_string()}")
print(f"  Churn rate: {churn_counts[1]/len(df)*100:.2f}%")

# ── Plot 1 : target distribution ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Task 1 — Dataset Overview", fontsize=14, fontweight="bold")

axes[0].bar(["Retained (0)", "Churned (1)"], churn_counts.values,
            color=["#4C72B0", "#DD8452"], edgecolor="white", linewidth=1.2)
axes[0].set_title("Target Variable Distribution")
axes[0].set_ylabel("Count")
for bar, val in zip(axes[0].patches, churn_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha="center", fontsize=11)

num_cols = df.select_dtypes(include="number").columns.drop("churn")
corr = df[num_cols].corrwith(df["churn"]).sort_values()
corr.plot(kind="barh", ax=axes[1], color=["#DD8452" if v > 0 else "#4C72B0" for v in corr.values])
axes[1].set_title("Numerical Feature Correlation with Churn")
axes[1].set_xlabel("Pearson r")
axes[1].axvline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.savefig(f"{RESULTS}/task1_dataset_overview.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"\n[Saved] {RESULTS}/task1_dataset_overview.png")


# =============================================================================
# TASK 2 — Data Preprocessing
# =============================================================================
print("\n" + "="*60)
print("TASK 2 — Data Preprocessing")
print("="*60)

# ── Drop identifier ──────────────────────────────────────────────────────────
df = df.drop(columns=["customer_id"])

# ── Categorical encoding (Label Encoding) ────────────────────────────────────
cat_cols = ["region", "plan_type", "contract_type", "payment_method"]
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le
    print(f"  Encoded '{col}': {list(le.classes_)}")

# ── Features / target split ──────────────────────────────────────────────────
X = df.drop(columns=["churn"]).values
y = df["churn"].values

# ── Train / test split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"\n  Train size : {X_train.shape[0]}  |  Test size : {X_test.shape[0]}")

# ── Standard scaling ─────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
print("  Numerical features scaled with StandardScaler.")

# ── Class weights (handle imbalance) ─────────────────────────────────────────
n_neg, n_pos = np.bincount(y_train)
class_weight = {0: 1.0, 1: n_neg / n_pos}
print(f"\n  Class imbalance → class_weight = {class_weight}")

INPUT_DIM = X_train.shape[1]


# =============================================================================
# TASK 3 & 4 — Baseline Neural Network + Evaluation
# =============================================================================
print("\n" + "="*60)
print("TASK 3 & 4 — Baseline Model & Evaluation")
print("="*60)

def build_model(hidden_layers=[(64, "relu"), (32, "relu")],
                learning_rate=0.001, input_dim=INPUT_DIM):
    """Build a configurable feed-forward neural network."""
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    for units, activation in hidden_layers:
        model.add(layers.Dense(units, activation=activation))
        model.add(layers.Dropout(0.2))
    model.add(layers.Dense(1, activation="sigmoid"))   # binary output
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")]
    )
    return model


def train_and_evaluate(model, X_tr, y_tr, X_te, y_te,
                       epochs=50, batch_size=32, class_weight=class_weight,
                       label="Baseline"):
    history = model.fit(
        X_tr, y_tr,
        validation_split=0.15,
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight,
        verbose=0
    )
    loss, acc, auc = model.evaluate(X_te, y_te, verbose=0)
    y_pred_prob = model.predict(X_te, verbose=0).ravel()
    y_pred      = (y_pred_prob >= 0.5).astype(int)
    cm          = confusion_matrix(y_te, y_pred)
    print(f"\n  [{label}]  Test Loss={loss:.4f}  Acc={acc:.4f}  AUC={auc:.4f}")
    print(f"  Classification report:\n{classification_report(y_te, y_pred, digits=3)}")
    return history, acc, auc, cm, y_pred_prob


# ── Train baseline ───────────────────────────────────────────────────────────
baseline_model = build_model()
baseline_model.summary()

hist_base, acc_base, auc_base, cm_base, prob_base = train_and_evaluate(
    baseline_model, X_train, y_train, X_test, y_test, label="Baseline")

# ── Plot training curves & confusion matrix ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Task 4 — Baseline Model Evaluation", fontsize=14, fontweight="bold")

epochs_range = range(1, len(hist_base.history["loss"]) + 1)

axes[0].plot(epochs_range, hist_base.history["loss"],     label="Train Loss")
axes[0].plot(epochs_range, hist_base.history["val_loss"], label="Val Loss", linestyle="--")
axes[0].set_title("Loss over Epochs"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(epochs_range, hist_base.history["accuracy"],     label="Train Acc")
axes[1].plot(epochs_range, hist_base.history["val_accuracy"], label="Val Acc", linestyle="--")
axes[1].set_title("Accuracy over Epochs"); axes[1].set_xlabel("Epoch"); axes[1].legend()

disp = ConfusionMatrixDisplay(cm_base, display_labels=["Retained", "Churned"])
disp.plot(ax=axes[2], colorbar=False, cmap="Blues")
axes[2].set_title("Confusion Matrix (Baseline)")

plt.tight_layout()
plt.savefig(f"{RESULTS}/evaluation_outputs.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"\n[Saved] {RESULTS}/evaluation_outputs.png")


# =============================================================================
# TASK 5 — Hyperparameter Experiments
# =============================================================================
print("\n" + "="*60)
print("TASK 5 — Hyperparameter Experiments")
print("="*60)

experiments = [
    {
        "name"         : "Baseline",
        "hidden_layers": [(64, "relu"), (32, "relu")],
        "lr"           : 0.001,
        "batch_size"   : 32,
        "epochs"       : 50,
    },
    {
        "name"         : "Exp-1: Deeper Network",
        "hidden_layers": [(128, "relu"), (64, "relu"), (32, "relu")],
        "lr"           : 0.001,
        "batch_size"   : 32,
        "epochs"       : 50,
    },
    {
        "name"         : "Exp-2: High LR",
        "hidden_layers": [(64, "relu"), (32, "relu")],
        "lr"           : 0.01,
        "batch_size"   : 32,
        "epochs"       : 50,
    },
    {
        "name"         : "Exp-3: Low LR + More Epochs",
        "hidden_layers": [(64, "relu"), (32, "relu")],
        "lr"           : 0.0001,
        "batch_size"   : 32,
        "epochs"       : 100,
    },
    {
        "name"         : "Exp-4: Large Batch",
        "hidden_layers": [(64, "relu"), (32, "relu")],
        "lr"           : 0.001,
        "batch_size"   : 128,
        "epochs"       : 50,
    },
    {
        "name"         : "Exp-5: Tanh Activation",
        "hidden_layers": [(64, "tanh"), (32, "tanh")],
        "lr"           : 0.001,
        "batch_size"   : 32,
        "epochs"       : 50,
    },
]

results_table = []

for exp in experiments:
    tf.random.set_seed(42); np.random.seed(42)
    m = build_model(hidden_layers=exp["hidden_layers"], learning_rate=exp["lr"])
    h = m.fit(
        X_train, y_train,
        validation_split=0.15,
        epochs=exp["epochs"],
        batch_size=exp["batch_size"],
        class_weight=class_weight,
        verbose=0
    )
    loss, acc, auc = m.evaluate(X_test, y_test, verbose=0)
    train_acc = h.history["accuracy"][-1]
    train_loss = h.history["loss"][-1]
    y_pred = (m.predict(X_test, verbose=0).ravel() >= 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    row = {
        "Configuration"      : exp["name"],
        "Hidden Layers"      : str([f"{u}×{a}" for u, a in exp["hidden_layers"]]),
        "LR"                 : exp["lr"],
        "Batch"              : exp["batch_size"],
        "Epochs"             : exp["epochs"],
        "Train Acc"          : round(train_acc, 4),
        "Test Acc"           : round(acc, 4),
        "Test AUC"           : round(auc, 4),
        "Test Loss"          : round(loss, 4),
        "TP / FP / FN / TN"  : f"{tp}/{fp}/{fn}/{tn}",
    }
    results_table.append(row)
    print(f"  ✓ {exp['name']:35s}  Acc={acc:.4f}  AUC={auc:.4f}")

# ── Save comparison CSV ──────────────────────────────────────────────────────
comparison_df = pd.DataFrame(results_table)
comparison_df.to_csv(f"{RESULTS}/model_comparison_table.csv", index=False)
print(f"\n[Saved] {RESULTS}/model_comparison_table.csv")

# ── Plot comparison table as image ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Task 5 — Hyperparameter Experiment Comparison", fontsize=14, fontweight="bold")

configs  = [r["Configuration"] for r in results_table]
test_acc = [r["Test Acc"]  for r in results_table]
test_auc = [r["Test AUC"]  for r in results_table]
x = np.arange(len(configs))
w = 0.35

bars1 = axes[0].bar(x - w/2, test_acc, w, label="Test Accuracy", color="#4C72B0")
bars2 = axes[0].bar(x + w/2, test_auc, w, label="Test AUC",      color="#DD8452")
axes[0].set_xticks(x); axes[0].set_xticklabels(configs, rotation=25, ha="right", fontsize=8)
axes[0].set_ylim(0, 1.1); axes[0].legend()
axes[0].set_title("Accuracy & AUC by Configuration")
axes[0].set_ylabel("Score")
for bar in bars1: axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                                f"{bar.get_height():.3f}", ha="center", fontsize=7)
for bar in bars2: axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                                f"{bar.get_height():.3f}", ha="center", fontsize=7)

# Table panel
col_labels = ["Config", "Test Acc", "Test AUC", "LR", "Batch", "Epochs"]
table_data  = [[r["Configuration"], r["Test Acc"], r["Test AUC"],
                r["LR"], r["Batch"], r["Epochs"]] for r in results_table]
axes[1].axis("off")
tbl = axes[1].table(cellText=table_data, colLabels=col_labels,
                     cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.6)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#4C72B0"); cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#EEF2FF")
axes[1].set_title("Comparison Summary Table", pad=12)

plt.tight_layout()
plt.savefig(f"{RESULTS}/model_comparison_table.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"[Saved] {RESULTS}/model_comparison_table.png")

print("\n" + "="*60)
print("All tasks complete! Results saved to ./results/")
print("="*60)